# Hispasonic ETL — Unified Dataset

**Source:** 12 CSV scrapes from Hispasonic (2022–2024)  
**Goal:** Load, inspect, clean and consolidate into a single homogeneous dataset  
**Output:** `../data/processed/hispasonic_unified.csv`

---

## Pipeline
1. Load all CSVs
2. Inspect columns and detect inconsistencies
3. Normalize to common schema (16 core columns)
4. Basic cleaning (types, nulls, duplicates)
5. Save unified dataset

## 1. Imports and config

In [1]:
import pandas as pd
import os
import glob

RAW_DIR = '../data/raw/'
PROCESSED_DIR = '../data/processed/'

# Core columns — common denominator across all files
CORE_COLUMNS = [
    'urgent', 'buy', 'change', 'sell', 'price', 'gift',
    'search', 'repair', 'parts', 'synt_brand', 'description',
    'city', 'published', 'expire', 'date_scrapped', 'seen'
]

print('pandas version:', pd.__version__)
print('Raw dir:', os.path.abspath(RAW_DIR))

pandas version: 2.2.3
Raw dir: /home/ion/dev/github/hispasonic/data/raw


## 2. Load and inspect each CSV

In [2]:
csv_files = sorted(glob.glob(os.path.join(RAW_DIR, '*.csv')))
print(f'{len(csv_files)} files found:\n')

for f in csv_files:
    df = pd.read_csv(f, nrows=0)
    cols = list(df.columns)
    extra = [c for c in cols if c not in CORE_COLUMNS]
    missing = [c for c in CORE_COLUMNS if c not in cols]
    print(f'{os.path.basename(f)}')
    print(f'  total cols: {len(cols)}')
    print(f'  extra cols: {extra}')
    print(f'  missing core cols: {missing}')
    print()

12 files found:

2022_08_26.csv
  total cols: 17
  extra cols: ['user']
  missing core cols: []

2022_08_30.csv
  total cols: 17
  extra cols: ['user']
  missing core cols: []

2022_09_01.csv
  total cols: 17
  extra cols: ['anon_user']
  missing core cols: []

2022_10_01.csv
  total cols: 18
  extra cols: ['user', 'anon_user']
  missing core cols: []

2022_11_01.csv
  total cols: 18
  extra cols: ['user', 'anon_user']
  missing core cols: []

2022_12_31.csv
  total cols: 18
  extra cols: ['user', 'anon_user']
  missing core cols: []

2023_03_01.csv
  total cols: 18
  extra cols: ['user', 'anon_user']
  missing core cols: []

2023_06_04.csv
  total cols: 18
  extra cols: ['user', 'anon_user']
  missing core cols: []

2023_06_07.csv
  total cols: 18
  extra cols: ['user', 'anon_user']
  missing core cols: []

2023_08_07.csv
  total cols: 18
  extra cols: ['user', 'anon_user']
  missing core cols: []

hpw2024517.csv
  total cols: 17
  extra cols: ['Unnamed: 0']
  missing core cols: []

h

## 3. Load all CSVs and normalize to core schema

In [3]:
frames = []

for f in csv_files:
    df = pd.read_csv(f)
    
    # Drop unnamed index column if present
    unnamed = [c for c in df.columns if c.startswith('Unnamed')]
    df = df.drop(columns=unnamed)
    
    # Keep only core columns that exist in this file
    available = [c for c in CORE_COLUMNS if c in df.columns]
    df = df[available]
    
    # Tag source file for traceability
    df['source_file'] = os.path.basename(f)
    
    frames.append(df)
    print(f'{os.path.basename(f)}: {len(df)} rows, {len(df.columns)} cols')

print(f'\nTotal frames loaded: {len(frames)}')

2022_08_26.csv: 764 rows, 17 cols
2022_08_30.csv: 760 rows, 17 cols
2022_09_01.csv: 805 rows, 17 cols
2022_10_01.csv: 837 rows, 17 cols
2022_11_01.csv: 784 rows, 17 cols
2022_12_31.csv: 264 rows, 17 cols
2023_03_01.csv: 270 rows, 17 cols
2023_06_04.csv: 261 rows, 17 cols
2023_06_07.csv: 261 rows, 17 cols
2023_08_07.csv: 425 rows, 17 cols
hpw2024517.csv: 264 rows, 17 cols
hpw2024526.csv: 267 rows, 17 cols

Total frames loaded: 12


## 4. Concatenate into unified dataset

In [4]:
df_all = pd.concat(frames, ignore_index=True)

print(f'Unified shape: {df_all.shape}')
print(f'Columns: {list(df_all.columns)}')
df_all.head()

Unified shape: (5962, 17)
Columns: ['urgent', 'buy', 'change', 'sell', 'price', 'gift', 'search', 'repair', 'parts', 'synt_brand', 'description', 'city', 'published', 'expire', 'date_scrapped', 'seen', 'source_file']


,urgent,buy,change,sell,price,gift,search,repair,parts,synt_brand,description,city,published,expire,date_scrapped,seen,source_file
0,0,0,0,1,350,0,0,0,0,dreadbox,dreadbox nyx v1,Valencia,2022/05/08,2022/09/23,2022/08/26,668,2022_08_26.csv
1,0,0,0,1,350,0,0,0,0,modal electronics,cobalt 5s,Castellón,2022/07/25,2022/09/23,2022/08/26,312,2022_08_26.csv
2,0,0,0,1,900,0,0,0,0,moog,moog little phatty stage 2,Baleares,2021/07/10,2022/09/28,2022/08/26,548,2022_08_26.csv
3,0,0,0,1,420,0,0,0,0,doepfer,último precio doepfer a-100 6u,Málaga,2022/07/01,2022/08/30,2022/08/26,272,2022_08_26.csv
4,0,0,0,1,480,0,0,0,0,mpc,mpc live ssd 250 envío incluido,Albacete,2022/08/26,2022/10/25,2022/08/26,78,2022_08_26.csv


## 5. Inspect nulls and data types

In [5]:
print('=== Null counts ===')
print(df_all.isnull().sum())
print()
print('=== Data types ===')
print(df_all.dtypes)
print()
print('=== Basic stats ===')
df_all.describe(include='all')

=== Null counts ===
urgent           0
buy              0
change           0
sell             0
price            0
gift             0
search           0
repair           0
parts            0
synt_brand       0
description      3
city             0
published        0
expire           0
date_scrapped    0
seen             0
source_file      0
dtype: int64

=== Data types ===
urgent            int64
buy               int64
change            int64
sell              int64
price             int64
gift              int64
search            int64
repair            int64
parts             int64
synt_brand       object
description      object
city             object
published        object
expire           object
date_scrapped    object
seen              int64
source_file      object
dtype: object

=== Basic stats ===


,urgent,buy,change,sell,price,gift,search,repair,parts,synt_brand,description,city,published,expire,date_scrapped,seen,source_file
count,5962.000000,5962.000000,5962.000000,5962.000000,5962.000000,5962.000000,5962.000000,5962.000000,5962.000000,5962,5959,5962,5962,5962,5962,5962.000000,5962
unique,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,249,2477,49,732,302,12,NaN,12
top,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-,arturia matrixbrute,Madrid,2022/10/11,2022/09/25,2022/10/01,NaN,2022_10_01.csv
freq,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1532,15,1461,82,138,837,NaN,837
mean,0.014425,0.039584,0.040926,0.919155,597.706978,0.002684,0.001677,0.002013,0.002516,NaN,NaN,NaN,NaN,NaN,NaN,790.006541,NaN
std,0.119243,0.194996,0.198135,0.272620,3078.973986,0.051739,0.040924,0.044822,0.050100,NaN,NaN,NaN,NaN,NaN,NaN,1482.982508,NaN
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,8.000000,NaN
25%,0.000000,0.000000,0.000000,1.000000,50.000000,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,190.000000,NaN
50%,0.000000,0.000000,0.000000,1.000000,220.000000,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,387.000000,NaN
75%,0.000000,0.000000,0.000000,1.000000,540.000000,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,823.000000,NaN


## 6. Fix data types

In [7]:
# Date columns
for col in ['published', 'expire', 'date_scrapped']:
    df_all[col] = pd.to_datetime(df_all[col], errors='coerce')

# Numeric columns
df_all['price'] = pd.to_numeric(df_all['price'], errors='coerce')
df_all['seen'] = pd.to_numeric(df_all['seen'], errors='coerce').astype('Int64')

# Boolean-like columns (should be 0/1)
bool_cols = ['urgent', 'buy', 'change', 'sell', 'gift', 'search', 'repair', 'parts']
for col in bool_cols:
    df_all[col] = pd.to_numeric(df_all[col], errors='coerce').astype('Int64')

print('Types after conversion:')
print(df_all.dtypes)

Types after conversion:
urgent                    Int64
buy                       Int64
change                    Int64
sell                      Int64
price                     int64
gift                      Int64
search                    Int64
repair                    Int64
parts                     Int64
synt_brand               object
description              object
city                     object
published        datetime64[ns]
expire           datetime64[ns]
date_scrapped    datetime64[ns]
seen                      Int64
source_file              object
dtype: object


## 7. Check for duplicate rows

In [8]:
subset_cols = ['description', 'price', 'published', 'city']
n_dupes = df_all.duplicated(subset=subset_cols).sum()
print(f'Duplicate rows (by {subset_cols}): {n_dupes}')

# Show duplicates if any
if n_dupes > 0:
    df_all[df_all.duplicated(subset=subset_cols, keep=False)].sort_values('description').head(20)

Duplicate rows (by ['description', 'price', 'published', 'city']): 2059


## 8. Save unified dataset

In [9]:
os.makedirs(PROCESSED_DIR, exist_ok=True)
output_path = os.path.join(PROCESSED_DIR, 'hispasonic_unified.csv')

df_all.to_csv(output_path, index=False)

print(f'Saved: {output_path}')
print(f'Shape: {df_all.shape}')
print(f'Date range: {df_all["date_scrapped"].min()} → {df_all["date_scrapped"].max()}')

Saved: ../data/processed/hispasonic_unified.csv
Shape: (5962, 17)
Date range: 2022-08-26 00:00:00 → 2024-05-26 00:00:00
